# Baseload hedge for the retail book — corrected

Each fix is marked **Fix N** (numbers refer to `mock_12_solution.md`).

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

In [2]:
df = pd.read_csv("../../data/hourly_power_clean.csv", parse_dates=["time"]).set_index("time")
SHARE = 0.01
TARIFF = 125.0
df["load"] = df["consumption_mwh"] * SHARE
df["cost"] = df["price_eur_mwh"] * df["load"]
df["revenue"] = TARIFF * df["load"]

## Monthly settlement frame

**Fix 6 (mean × sum).** Monthly cost is the sum of hourly `price * load`, not the monthly average
price times the monthly load. Expensive hours are high-load hours, so the load-weighted price is
systematically above the simple average.

In [3]:
monthly = pd.DataFrame({
    "avg_price": df["price_eur_mwh"].resample("MS").mean(),
    "load": df["load"].resample("MS").sum(),
    "hours": df["load"].resample("MS").size(),
    "cost": df["cost"].resample("MS").sum(),
})
monthly["cost_mean_x_sum"] = monthly["avg_price"] * monthly["load"]
monthly["lw_price"] = monthly["cost"] / monthly["load"]
print("load-weighted / average price:",
      (monthly["lw_price"] / monthly["avg_price"]).describe()[["mean", "min", "max"]].round(3).to_dict())
print(f"understatement of 2-year cost: {(1 - monthly.cost_mean_x_sum.sum()/monthly.cost.sum())*100:.1f}%")

load-weighted / average price: {'mean': 1.027, 'min': 1.014, 'max': 1.044}
understatement of 2-year cost: 2.4%


## A forward price you could actually trade at

**Fix 4 (look-ahead forward).** `df.price.mean()` over 2022–2023 is not a forward price: nobody
could buy at the realised two-year average in January 2022. The hedge must be struck at a price
known *before* the month. Absent a forward curve in the data, the previous month's average is a
crude but honest proxy. The quantity at risk is the *surprise*: realised cost minus what the
book would have cost at the forward.

In [4]:
V = df["load"].mean()                                    # baseload MW
monthly["F"] = monthly["avg_price"].shift(1)             # known at the start of the month
monthly["payoff_per_mw"] = (monthly["avg_price"] - monthly["F"]) * monthly["hours"]
monthly["surprise"] = monthly["cost"] - monthly["F"] * monthly["load"]
monthly = monthly.dropna()
print(len(monthly), "settled months")
monthly[["avg_price", "F", "lw_price", "surprise"]].round(0).head()

23 settled months


,avg_price,F,lw_price,surprise
time,,,,
2022-02-01 00:00:00+00:00,101.0,103.0,104.0,17148.0
2022-03-01 00:00:00+00:00,98.0,101.0,101.0,-136937.0
2022-04-01 00:00:00+00:00,81.0,98.0,84.0,-3008732.0
2022-05-01 00:00:00+00:00,96.0,81.0,99.0,3554296.0
2022-06-01 00:00:00+00:00,102.0,96.0,105.0,1623975.0


## Hedge ratio: fit on 2022, test on 2023

**Fix 1 (levels vs surprises, hourly vs monthly).** The original regressed hourly cost *levels* on
hourly price levels. That slope (1.22) mostly measures that expensive hours carry more load,
i.e. intra-day shape, plus the 2022 trend. Shape is not hedged by over-buying baseload (it would
leave the book long every night); it is hedged with shaped products, or accepted. The decision
variable is monthly settlement risk, so fit on monthly surprises.

**Fix 2 (ddof).** Use one estimator: `pandas .cov()` and `.var()` are both `ddof=1`.

**Fix 3 (in-sample effectiveness).** Fit on 2022, report effectiveness on 2023.

In [5]:
train = monthly.loc["2022"]
test = monthly.loc["2023"]
h_mw = train["surprise"].cov(train["payoff_per_mw"]) / train["payoff_per_mw"].var()
h_ratio = h_mw / V
print(f"hedge volume {h_mw:.1f} MW = {h_ratio:.3f} x average load ({V:.1f} MW)")

def effectiveness(h_mw, frame):
    hedged = frame["surprise"] - h_mw * frame["payoff_per_mw"]
    return 1 - hedged.var() / frame["surprise"].var(), hedged.std()

rows = []
for label, hm in [("unhedged", 0.0), ("1.0 x load", V), ("fitted 2022", h_mw), ("1.22 x load (original)", 1.22 * V)]:
    eff_tr, sd_tr = effectiveness(hm, train)
    eff_te, sd_te = effectiveness(hm, test)
    rows.append({"hedge": label, "MW": round(hm, 1), "eff 2022 (fit)": round(eff_tr, 3),
                 "eff 2023 (OOS)": round(eff_te, 3), "residual sd 2023 (kEUR/month)": round(sd_te / 1e3)})
pd.DataFrame(rows).set_index("hedge")

hedge volume 293.8 MW = 1.002 x average load (293.2 MW)


,MW,eff 2022 (fit),eff 2023 (OOS),residual sd 2023 (kEUR/month)
hedge,,,,
unhedged,0.0,0.000,0.000,2167
1.0 x load,293.2,0.998,0.997,111
fitted 2022,293.8,0.998,0.997,111
1.22 x load (original),357.6,0.951,0.949,489


**Fix 11 (residual in money).** "Residual sd ≈ 1,200 EUR" in the original was the sd of an
*hourly* cost residual. Risk is carried to settlement: the monthly figure above is the one to
compare with the book's margin.

In [6]:
monthly["margin"] = TARIFF * monthly["load"] - monthly["cost"]
print(f"average monthly margin: {monthly['margin'].mean()/1e3:,.0f} kEUR")
print(f"unhedged surprise sd:   {test['surprise'].std()/1e3:,.0f} kEUR/month")
print(f"1.0x hedged residual sd:{effectiveness(V, test)[1]/1e3:,.0f} kEUR/month")

average monthly margin: 5,029 kEUR
unhedged surprise sd:   2,167 kEUR/month
1.0x hedged residual sd:111 kEUR/month


## Seasonality

**Fix 10 (pooled months).** `groupby(index.month)` pools January 2022 with January 2023 into 12
rows. There are 24 months; group by year *and* month if that is what the text claims.

In [7]:
season = df.groupby([df.index.year, df.index.month])["cost"].sum().unstack(0)
season.columns = [f"cost_{c}" for c in season.columns]
season.round(0).T

time,1,2,3,4,5,6,7,8,9,10,11,12
cost_2022,25565084.0,22031285.0,23431042.0,17338200.0,20457702.0,20538874.0,24136132.0,25014585.0,25933853.0,28756935.0,32483512.0,29769961.0
cost_2023,28976714.0,24012309.0,23734328.0,17838708.0,15454710.0,12066031.0,13270621.0,14668999.0,14302418.0,18664517.0,20888965.0,22357930.0


## Scenario VaR

**Fix 8 (regimes and the tail).** Drawing days from both years mixes the 2022 gas spike into every
"2023" month. Bootstrap within the year you are pricing. VaR is a *loss* quantile: with P&L
defined as margin, the 95% VaR is `-percentile(pnl, 5)`; `percentile(pnl, 95)` is the best case.

**Fix 9 (sign).** The forward pays `(spot - F) * V * hours` when spot rises, which is when cost
rises, so it *adds* to margin: `margin_hedged = margin + h * payoff`. The original subtracted it.

In [8]:
rng = np.random.default_rng(0)
d23 = df.loc["2023"]
days = [g for _, g in d23.groupby(d23.index.date)]
F_scen = df.loc["2022-12", "price_eur_mwh"].mean()         # forward known before 2023

def simulate(h_mw, n=2000):
    out = []
    for _ in range(n):
        m = pd.concat([days[i] for i in rng.integers(0, len(days), 30)])
        margin = m["revenue"].sum() - m["cost"].sum()
        payoff = (m["price_eur_mwh"].mean() - F_scen) * h_mw * len(m)
        out.append(margin + payoff)
    pnl = np.array(out)
    var95 = -np.percentile(pnl, 5)
    cvar95 = -pnl[pnl <= np.percentile(pnl, 5)].mean()
    return pnl.mean(), pnl.std(), var95, cvar95

rows = {lab: simulate(hm) for lab, hm in [("unhedged", 0.0), ("1.0 x load", V), ("1.22 x load", 1.22 * V)]}
pd.DataFrame(rows, index=["mean margin", "sd", "95% VaR (loss)", "95% CVaR (loss)"]).T.round(0)

,mean margin,sd,95% VaR (loss),95% CVaR (loss)
unhedged,7736093.0,775422.0,-6410908.0,-6085731.0
1.0 x load,-1031691.0,124744.0,1245946.0,1297079.0
1.22 x load,-2961330.0,297257.0,3473802.0,3597783.0


The hedged book has a *negative* mean here: the proxy forward (December 2022 average, ~120) was far
above realised 2023 spot, so locking it in cost money relative to floating. A hedge fixes the
price, it does not improve the expected margin; in practice the tariff is set off the same forward
curve. What the hedge does is visible in the `sd` column: 775 kEUR down to 125 kEUR.

## Results

In [9]:
eff_te, sd_te = effectiveness(V, test)
print(f"hedge ratio (fitted on 2022 surprises): {h_ratio:.2f} x load")
print(f"OOS effectiveness 2023 at 1.0x:          {eff_te:.3f}")
print(f"residual sd at 1.0x:                     {sd_te/1e3:,.0f} kEUR/month")
print(f"OOS effectiveness at 1.22x:              {effectiveness(1.22*V, test)[0]:.3f}")

hedge ratio (fitted on 2022 surprises): 1.00 x load
OOS effectiveness 2023 at 1.0x:          0.997
residual sd at 1.0x:                     111 kEUR/month
OOS effectiveness at 1.22x:              0.949


A 1.0x baseload hedge struck at a price known in advance removes almost all of the monthly price
risk in this data; 1.22x *adds* risk out of sample because the extra 0.22x is an outright long
position. The remaining risk is load-shape and volume risk, which a baseload strip cannot hedge
and which the original notebook assumed away by using realised load (**Fix 5**). Note that this
data has iid hourly price noise, so month-average surprises are unusually clean; with real
forward-vs-spot basis the effectiveness would be lower.